<div align="center">

<h1 style="color:blue; font-weight:bold;">
Incident Data Auditing (DfT Road Collision and Casualty Data)
</h1>

</div>

### Step 1: Import Required Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

### Step 2: Locate and Verify the Incident Files

Identify the directory containing the 2025 DfT road safety files and confirm that both expected files are present: the collision file (one row per collision) and the casualty file (one row per casualty). Checking file names and sizes before loading helps catch missing or incomplete downloads early.

In [5]:
# Define the directory containing the DfT road safety files.
incident_dir = Path("../data/incident")

# Match on the end of the file name. 
collision_files = sorted(incident_dir.glob("*-collision-2025.csv"))
casualty_files = sorted(incident_dir.glob("*-casualty-2025.csv"))

# Exactly one file of each type is expected.
print("Collision files found:", len(collision_files))
print("Casualty files found:", len(casualty_files))

for file in collision_files + casualty_files:
    # File size gives a quick check that the download completed.
    size_mb = file.stat().st_size / 1e6
    print(f"{file.name}: {size_mb:.1f} MB")

assert len(collision_files) == 1, "Expected exactly one 2025 collision file."
assert len(casualty_files) == 1, "Expected exactly one 2025 casualty file."

collision_path = collision_files[0]
casualty_path = casualty_files[0]

Collision files found: 1
Casualty files found: 1
dft-road-casualty-statistics-collision-2025.csv: 19.3 MB
dft-road-casualty-statistics-casualty-2025.csv: 10.3 MB


### Step 3: Load and Inspect the Collision and Casualty Data

Load both CSV files into pandas DataFrames and inspect their size and columns. The collision file holds one row per reported collision, with its date, time, location and road conditions. The casualty file holds one row per person injured, linked to its collision through `collision_index`.

`collision_index` and the LSOA codes are read as text. They are identifiers, not numbers, and reading them as numbers could drop leading zeros or mangle codes that contain letters.

In [6]:
# Identifier columns are read as text so they are never treated as numbers.
id_columns = {
    "collision_index": str,
    "collision_ref_no": str,
    "lsoa_of_accident_location": str,
    "lsoa_of_casualty": str,
}

# low_memory=False makes pandas read the whole file before choosing column
# types, which avoids mixed-type warnings in large files.
collisions = pd.read_csv(
    collision_path,
    dtype={k: v for k, v in id_columns.items() if k != "lsoa_of_casualty"},
    low_memory=False,
)
casualties = pd.read_csv(
    casualty_path,
    dtype={k: v for k, v in id_columns.items() if k != "lsoa_of_accident_location"},
    low_memory=False,
)

# Rows and columns show the scale of each table.
print("Collisions:", collisions.shape)
print("Casualties:", casualties.shape)

# List the columns to see which fields are available for the weather analysis.
print("\nCollision columns:")
print(collisions.columns.tolist())
print("\nCasualty columns:")
print(casualties.columns.tolist())

# Preview the fields most relevant to this project.
collisions[[
    "collision_index", "date", "time",
    "location_easting_osgr", "location_northing_osgr",
    "collision_severity", "weather_conditions", "road_surface_conditions",
]].head()

Collisions: (101525, 44)
Casualties: (127883, 24)

Collision columns:
['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail_historic', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag', 'lsoa_of_acciden

,collision_index,date,time,location_easting_osgr,location_northing_osgr,collision_severity,weather_conditions,road_surface_conditions
0,202517M101925,04/02/2025,15:00,447673.0,519279.0,3,1,1
1,202517L205225,07/05/2025,09:35,454611.0,518597.0,3,1,1
2,2025070971700,27/11/2025,18:18,354476.0,382656.0,3,1,2
3,2025070403274,15/05/2025,16:30,361793.0,393163.0,3,1,1
4,2025071000604,08/12/2025,07:45,366083.0,389327.0,3,1,2


### Step 4: Audit Dates, Identifiers and the Collision–Casualty Link

Before using the records, we need to confirm that they cover the expected period, that each collision and casualty appears only once, and that the two tables link correctly.

We will:
- Check that every collision is from 2025, and that all dates and times can be parsed in the formats stated in the DfT data guide (`DD/MM/YYYY` and `HH:MM`).
- Confirm that the recorded `day_of_week` agrees with the date.
- Check for duplicate collisions and duplicate casualties.
- Verify that every casualty links to a collision through `collision_index`, and that each collision's `number_of_casualties` matches the number of casualty records linked to it.

These checks determine whether the tables can be joined to each other, and to the daily weather data, without losing or double-counting records.

In [8]:
# --- Dates and times ---

# All records should belong to 2025, matching the weather data.
print("Collision years:", collisions["collision_year"].unique())

# Parse using the formats stated in the DfT data guide. errors="coerce" turns
# any value that does not fit the format into NaT, so failures can be counted.
collision_dates = pd.to_datetime(collisions["date"], format="%d/%m/%Y", errors="coerce")
collision_times = pd.to_datetime(collisions["time"], format="%H:%M", errors="coerce")

print("Unparseable dates:", int(collision_dates.isna().sum()))
print("Unparseable times:", int(collision_times.isna().sum()))
print("Date range:", collision_dates.min().date(), "to", collision_dates.max().date())
print("Distinct dates with collisions:", collision_dates.dt.date.nunique())

# The DfT codes day_of_week as 1 = Sunday ... 7 = Saturday. pandas numbers
# days as 0 = Monday ... 6 = Sunday, so we convert before comparing.
dft_day_from_date = (collision_dates.dt.dayofweek + 1) % 7 + 1
print("Rows where day_of_week disagrees with the date:",
      int((dft_day_from_date != collisions["day_of_week"]).sum()))

# --- Duplicates ---

# collision_index is documented as unique for each collision.
print("\nDuplicate collision_index values:",
      int(collisions["collision_index"].duplicated().sum()))

# A casualty is identified by its collision, vehicle and casualty reference.
casualty_key = ["collision_index", "vehicle_reference", "casualty_reference"]
print("Duplicate casualty records:",
      int(casualties.duplicated(subset=casualty_key).sum()))

# --- Link between the two tables ---

# Every casualty must belong to a collision in the collision file.
orphan_casualties = ~casualties["collision_index"].isin(collisions["collision_index"])
print("\nCasualties with no matching collision:", int(orphan_casualties.sum()))

# Every reported collision in STATS19 involves at least one casualty.
collisions_without_casualties = ~collisions["collision_index"].isin(casualties["collision_index"])
print("Collisions with no casualty records:", int(collisions_without_casualties.sum()))

# The count stored on each collision should equal its linked casualty rows.
linked_counts = casualties.groupby("collision_index").size()
recorded_counts = collisions.set_index("collision_index")["number_of_casualties"]
count_mismatch = recorded_counts != linked_counts.reindex(recorded_counts.index, fill_value=0)

print("Collisions where number_of_casualties disagrees with linked rows:",
      int(count_mismatch.sum()))
print("Total casualties recorded on collisions:", int(recorded_counts.sum()))
print("Total casualty records:", len(casualties))

Collision years: [2025]
Unparseable dates: 0
Unparseable times: 0
Date range: 2025-01-01 to 2025-12-31
Distinct dates with collisions: 365
Rows where day_of_week disagrees with the date: 0

Duplicate collision_index values: 0
Duplicate casualty records: 0

Casualties with no matching collision: 0
Collisions with no casualty records: 0
Collisions where number_of_casualties disagrees with linked rows: 0
Total casualties recorded on collisions: 127883
Total casualty records: 127883


#### Interpretation of Step 4's Results

| Check | Result | Interpretation |
|---|---|---|
| Collision year | 2025 only | Matches the weather data |
| Dates and times | All parsed; 1 Jan – 31 Dec 2025 | No format errors |
| Days with collisions | 365 | Every day of the year is represented |
| `day_of_week` vs date | 0 disagreements | Dates are internally consistent |
| Duplicate collisions | 0 | `collision_index` is a reliable unique key |
| Duplicate casualties | 0 | Each casualty is recorded once |
| Casualties without a collision | 0 | Casualties can be joined to collisions without loss |
| Collisions without casualties | 0 | Every collision has at least one casualty record |
| `number_of_casualties` vs linked rows | 0 mismatches; 127,883 on both sides | The two tables are fully consistent |

The collision and casualty tables are structurally sound and can be joined on `collision_index` without losing or double-counting records. Because every collision has a parseable date, each one can be matched to a daily weather value.

### Step 5: Audit Missing and Unknown Values Using the DfT Data Guide

In the DfT data, a missing value is usually not left blank. It is stored as a code: `-1` means "Data missing or out of range", and many fields also have an "unknown" code such as `9` or `99`. A standard blank-value check would therefore report almost nothing missing, even where information is absent.

To measure missingness correctly, we load the official DfT data guide and, for each field, collect the codes whose label marks the value as missing or unknown. We then count, for every column:
- blank values (`NaN`),
- values coded as missing or unknown according to the guide.

This shows which fields are reliable enough to use in the weather analysis, and which should be treated with caution or set aside.

In [10]:
# Load the code list from the official DfT data guide.
guide_files = sorted(incident_dir.glob("*data-guide*.xlsx"))
assert len(guide_files) == 1, "Expected exactly one DfT data guide in the incident folder."

code_list = pd.read_excel(guide_files[0], sheet_name="2024_code_list")

# The code column mixes numeric codes with format notes (e.g. "DD/MM/YYYY"),
# so convert it to numbers and keep only the numeric codes.
code_list["code"] = pd.to_numeric(code_list["code/format"], errors="coerce")

# A code counts as missing or unknown if its label says so, for example
# "Data missing or out of range", "Unknown", "unknown (self reported)" or "Not Known".
# Labels such as "Darkness - lighting unknown" are real categories, so the
# pattern only matches labels that start with "unknown" or mention missing or not known.
missing_label = code_list["label"].astype(str).str.contains(
    r"^\s*unknown|missing|not known", case=False, regex=True
)

missing_codes = (
    code_list[missing_label & code_list["code"].notna()]
    .groupby(["table", "field name"])["code"]
    .apply(set)
)


def missing_summary(df, table_name):
    """Count blank and coded-missing values for every column in one table."""
    rows = []
    for column in df.columns:
        codes = missing_codes.get((table_name, column), set())
        blank = int(df[column].isna().sum())
        coded = int(df[column].isin(codes).sum()) if codes else 0
        rows.append({
            "column": column,
            "blank": blank,
            "coded_missing_or_unknown": coded,
            "codes_used": sorted(codes) if codes else "",
            "pct_missing": round(100 * (blank + coded) / len(df), 2),
        })

    summary = pd.DataFrame(rows).set_index("column")
    # Filter on the counts, not the rounded percentage, so columns with only
    # a handful of missing values (e.g. 2 of 101,525) are not hidden by rounding.
    has_missing = (summary["blank"] + summary["coded_missing_or_unknown"]) > 0
    return summary[has_missing].sort_values("pct_missing", ascending=False)


print("Collision table: columns with missing or unknown values")
display(missing_summary(collisions, "collision"))

print("\nCasualty table: columns with missing or unknown values")
display(missing_summary(casualties, "casualty"))

Collision table: columns with missing or unknown values


,blank,coded_missing_or_unknown,codes_used,pct_missing
column,,,,
pedestrian_crossing_human_control_historic,0,88330,"[-1.0, 9.0]",87.00
pedestrian_crossing_physical_facilities_historic,0,88218,"[-1.0, 9.0]",86.89
junction_detail_historic,0,88146,"[-1.0, 99.0]",86.82
special_conditions_at_site,0,88084,"[-1.0, 9.0]",86.76
carriageway_hazards_historic,0,88038,"[-1.0, 9.0]",86.72
junction_control,0,48013,"[-1.0, 9.0]",47.29
second_road_number,0,25105,[-1.0],24.73
junction_detail,0,8128,"[-1.0, 99.0]",8.01
trunk_road_flag,0,7108,[-1.0],7.00



Casualty table: columns with missing or unknown values


,blank,coded_missing_or_unknown,codes_used,pct_missing
column,,,,
pedestrian_road_maintenance_worker,0,17502,"[-1.0, 2.0]",13.69
enhanced_casualty_severity,0,17394,[-1.0],13.60
casualty_distance_banding,0,14663,[-1.0],11.47
casualty_imd_decile,0,14663,[-1.0],11.47
pedestrian_movement,0,5608,"[-1.0, 9.0]",4.39
age_of_casualty,0,2741,[-1.0],2.14
age_band_of_casualty,0,2741,[-1.0],2.14
pedestrian_location,0,1901,"[-1.0, 10.0]",1.49
sex_of_casualty,0,1372,"[-1.0, 9.0]",1.07


#### Interpretation of Step 5

Missing values in the DfT data are almost entirely coded (`-1`, `9`, `99`) rather than blank. Only the four location columns contain blanks: 2 collisions have no easting, northing, longitude or latitude. A blank-only check would have reported the data as nearly complete.

**Fields needed for this project are close to complete:**

| Field | Missing or unknown | Use in this project |
|---|---:|---|
| Location (easting, northing) | 2 collisions | Matching each collision to a weather grid cell |
| `date`, `time` | 0 | Matching each collision to a weather day |
| `collision_severity`, `casualty_severity` | 0 | Outcome measures |
| `weather_conditions` | 2.35% | Police-recorded weather, compared with HadUK-Grid |
| `road_surface_conditions` | 1.95% | Police-recorded surface (wet, ice), compared with rainfall and temperature |
| `light_conditions` | 0.02% | Separating daylight and darkness |
| `age_of_casualty`, `sex_of_casualty` | 2.14%, 1.07% | Describing who is affected |

**Fields set aside:**
- The `_historic` fields and `special_conditions_at_site` are about 87% missing. DfT revised these codes, and the historic versions are filled in for only part of the data, so the current fields (`junction_detail`, `pedestrian_crossing`, `carriageway_hazards`) are used instead where needed.
- `junction_control` (47% missing) and `second_road_number` (25%) are not needed for the weather analysis.
- `casualty_imd_decile` and `casualty_distance_banding` are both missing for exactly 14,663 casualties. Both are derived from the casualty's home postcode, which suggests that postcode was unavailable for those casualties. They are excluded from the core analysis and noted as a limitation if used.

Records with unknown weather or road surface will be kept for the collision–weather match, since the HadUK-Grid data supplies the weather independently. They will be excluded only from comparisons that use the police-recorded fields.

### Step 6: Audit Collision Locations and Geographic Coverage

Each collision must be placed in a 5 km HadUK-Grid cell, so its location must be present, valid and in the same coordinate system as the weather data.

We will:
- Identify the collisions without a recorded location.
- Check that eastings and northings fall within the British National Grid (EPSG:27700), the CRS of the HadUK-Grid data.
- Cross-check the eastings and northings against the separately recorded longitude and latitude by converting one into the other. Close agreement confirms that the coordinates are reliable.
- Confirm geographic coverage using police force codes. The DfT data covers Great Britain only (England, Wales and Scotland), so Northern Ireland is expected to be absent.

In [11]:
from pyproj import Transformer

# --- Missing locations ---

no_location = collisions["location_easting_osgr"].isna() | collisions["location_northing_osgr"].isna()
print("Collisions without a location:", int(no_location.sum()))
display(collisions.loc[no_location, ["collision_index", "date", "police_force", "collision_severity"]])

located = collisions[~no_location]

# --- British National Grid range ---

# British National Grid covers roughly 0-700 km east and 0-1,300 km north.
# Values outside this range would indicate a coding error.
print("\nEasting range:", located["location_easting_osgr"].min(), "to",
      located["location_easting_osgr"].max())
print("Northing range:", located["location_northing_osgr"].min(), "to",
      located["location_northing_osgr"].max())

outside_bng = ~(
    located["location_easting_osgr"].between(0, 700000)
    & located["location_northing_osgr"].between(0, 1300000)
)
print("Locations outside the British National Grid extent:", int(outside_bng.sum()))

# --- Cross-check eastings/northings against longitude/latitude ---

# Convert the recorded longitude and latitude (EPSG:4326) to British National
# Grid and measure the distance to the recorded easting and northing.
# always_xy=True keeps the order as (longitude, latitude) -> (easting, northing).
to_bng = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)
converted_e, converted_n = to_bng.transform(
    located["longitude"].to_numpy(), located["latitude"].to_numpy()
)
offset_m = np.hypot(
    converted_e - located["location_easting_osgr"].to_numpy(),
    converted_n - located["location_northing_osgr"].to_numpy(),
)

print(f"\nOffset between the two location fields: median {np.median(offset_m):.2f} m, "
      f"max {offset_m.max():.2f} m")
print("Collisions where the two fields disagree by more than 100 m:", int((offset_m > 100).sum()))

# --- Geographic coverage ---

# DfT police force codes: 1-59 England, 60-63 Wales, 99 Police Scotland.
# Northern Ireland is not included in the DfT road safety data.
force = collisions["police_force"]
coverage = pd.Series({
    "England": int((force < 60).sum()),
    "Wales": int(force.between(60, 63).sum()),
    "Scotland": int((force == 99).sum()),
    "Other codes": int((~((force < 60) | force.between(60, 63) | (force == 99))).sum()),
})
print("\nCollisions by country (from police force code):")
print(coverage)

# --- Local authority fields ---

# Check the two local authority fields before relying on either for mapping.
print("\nlocal_authority_district values:", collisions["local_authority_district"].unique()[:5])
print("local_authority_ons_district examples:",
      collisions["local_authority_ons_district"].head(3).tolist())

Collisions without a location: 2


,collision_index,date,police_force,collision_severity
66270,2025622500721,04/11/2025,62,3
86596,2025622500752,06/11/2025,62,2



Easting range: 71253.0 to 655126.0
Northing range: 10237.0 to 1162591.0
Locations outside the British National Grid extent: 0

Offset between the two location fields: median 0.35 m, max 0.65 m
Collisions where the two fields disagree by more than 100 m: 0

Collisions by country (from police force code):
England        94418
Wales           3003
Scotland        4104
Other codes        0
dtype: int64

local_authority_district values: [-1]
local_authority_ons_district examples: ['E06000002', 'E06000003', 'E06000006']


#### Interpretation of Step 6

| Check | Result | Interpretation |
|---|---|---|
| Collisions without a location | 2 (both Dyfed-Powys Police, November 2025) | Cannot be placed on the weather grid; excluded from the spatial join |
| Easting / northing range | 71,253–655,126 m E; 10,237–1,162,591 m N | All within British National Grid |
| Easting/northing vs longitude/latitude | Median offset 0.35 m, max 0.65 m | The two location fields agree; the British National Grid coordinates can be used directly |
| Coverage | England 94,418; Wales 3,003; Scotland 4,104; other 0 | Great Britain only, as documented |
| `local_authority_district` | -1 for every row | No longer populated; `local_authority_ons_district` (ONS codes such as E06000002) is used instead |

The collision locations are reliable and use the same coordinate system as HadUK-Grid (EPSG:27700), so collisions can be placed in weather grid cells by arithmetic alone, with no reprojection.

Because the DfT data excludes Northern Ireland, the merged analysis covers **Great Britain**, not the whole UK. The weather dataset from notebook 01 includes Northern Ireland, so its cells will be excluded at the merge stage.

The `local_authority_district` field shows that a field can be unusable without being flagged by the data guide's missing codes, which is why key fields are also checked individually.

### Step 7: Decode Key Fields and Inspect Their Distributions

The DfT stores categorical fields as numeric codes. To interpret them, we translate the codes into the labels given in the official data guide for the fields this project uses: severity, police-recorded weather, road surface, light conditions and urban/rural area for collisions, and severity and casualty class for casualties.

This step also checks that every code in the data exists in the guide. A code not listed in the guide would suggest an error in the data or a mismatch between the data and the guide version.

The distributions give a first view of the data. For example, they show how often police recorded rain, and how severe collisions typically are.

In [12]:
def decode(df, table_name, field):
    """Translate one coded field into its labels from the DfT data guide."""
    # Build a code -> label lookup for this field only, since the same
    # code number means different things in different fields.
    field_codes = code_list[
        (code_list["table"] == table_name)
        & (code_list["field name"] == field)
        & code_list["code"].notna()
    ]
    lookup = dict(zip(field_codes["code"], field_codes["label"]))
    return df[field].map(lookup)


# Fields used in the weather analysis.
collision_fields = [
    "collision_severity",
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions",
    "urban_or_rural_area",
]
casualty_fields = ["casualty_severity", "casualty_class"]

# Add a readable label column next to each code, e.g. weather_conditions_label.
for field in collision_fields:
    collisions[f"{field}_label"] = decode(collisions, "collision", field)
for field in casualty_fields:
    casualties[f"{field}_label"] = decode(casualties, "casualty", field)

# Any code without a label is not in the data guide.
for df, fields, name in [(collisions, collision_fields, "Collision"),
                         (casualties, casualty_fields, "Casualty")]:
    for field in fields:
        unmatched = int(df[f"{field}_label"].isna().sum())
        print(f"{name} {field}: codes not found in the guide = {unmatched}")

# Show each distribution as a percentage of records.
for df, fields in [(collisions, collision_fields), (casualties, casualty_fields)]:
    for field in fields:
        print(f"\n{field}")
        print((df[f"{field}_label"].value_counts(normalize=True) * 100).round(2).to_string())

Collision collision_severity: codes not found in the guide = 0
Collision weather_conditions: codes not found in the guide = 0
Collision road_surface_conditions: codes not found in the guide = 0
Collision light_conditions: codes not found in the guide = 0
Collision urban_or_rural_area: codes not found in the guide = 0
Casualty casualty_severity: codes not found in the guide = 0
Casualty casualty_class: codes not found in the guide = 0

collision_severity
collision_severity_label
Slight     73.76
Serious    24.81
Fatal       1.43

weather_conditions
weather_conditions_label
Fine no high winds       82.15
Raining no high winds    10.08
Other                     3.26
Unknown                   2.35
Raining + high winds      0.87
Fine + high winds         0.70
Fog or mist               0.34
Snowing no high winds     0.23
Snowing + high winds      0.02

road_surface_conditions
road_surface_conditions_label
Dry                             74.54
Wet or damp                     21.98
unknown (se

#### Interpretation of Step 7

Every code in the fields used by this project matches an entry in the DfT data guide, so the labels can be trusted.

| Field | Main categories | Note for the analysis |
|---|---|---|
| Collision severity | Slight 73.76%, Serious 24.81%, Fatal 1.43% | Fatal collisions are rare, so fatal-only results will rest on small numbers |
| Weather (police-recorded) | Fine 82.85%, Rain 10.95%, Snow 0.25%, Fog 0.34% | Rain codes include "no high winds" (10.08%) and "+ high winds" (0.87%); "Other" (3.26%) gives no usable weather information |
| Road surface | Dry 74.54%, Wet or damp 21.98%, Frost or ice 1.24% | About twice as many collisions occur on wet roads as during reported rain, as roads stay wet after rain stops |
| Light | Daylight 71.72%, Darkness 28.26% | Darkness is a possible confounder: wet, cold weather is more common in winter, when nights are longer |
| Urban / rural | Urban 66.14%, Rural 33.86% | Allows urban and rural patterns to be compared |
| Casualty severity | Slight 77.09%, Serious 21.71%, Fatal 1.20% | Consistent with collision severity |
| Casualty class | Driver or rider 65.67%, Passenger 18.71%, Pedestrian 15.62% | Pedestrians are a vulnerable group worth separating |

The gap between reported rain (10.95%) and wet road surfaces (21.98%) shows why an independent rainfall measure is useful: the police record conditions at the moment of the collision, while HadUK-Grid records the total rainfall over the day.

### Step 8: Build the Cleaned Collision Dataset

We now produce the cleaned incident dataset that will be merged with the weather data in notebook 03. It has one row per collision, with each collision's casualties summarised as counts, so the spatial and daily weather join does not duplicate any collision.

The casualty table is not merged row by row. Doing so would repeat each collision once per casualty, inflating collision counts and giving multi-casualty collisions extra weight. Summarising casualties per collision first keeps one row per collision while preserving casualty totals.

The transformation:
1. Removes the 2 collisions without a location, since they cannot be placed on the weather grid.
2. Converts `date` and `time` into proper date-time values and adds `hour` and `month`.
3. Adds `country` from the police force code.
4. Creates true/false flags from the police-recorded conditions (rain, snow, wet surface, ice or snow). Unknown records are left blank, not set to false, so a missing report is never counted as "no rain".
5. Counts each collision's casualties by severity (fatal, serious, slight) and the number of pedestrian casualties, from the casualty table.
6. Keeps only the columns needed for the analysis.

In [13]:
# --- 1. Exclude collisions that cannot be placed on the weather grid ---------

has_location = collisions["location_easting_osgr"].notna() & collisions["location_northing_osgr"].notna()
collisions_clean = collisions[has_location].copy()
print("Collisions before:", len(collisions))
print("Collisions without a location removed:", int((~has_location).sum()))
print("Collisions kept:", len(collisions_clean))

# --- 2. Proper date and time types ------------------------------------------

# A real datetime lets later notebooks group by day, month or hour directly.
collisions_clean["date"] = pd.to_datetime(collisions_clean["date"], format="%d/%m/%Y")
collisions_clean["datetime"] = pd.to_datetime(
    collisions_clean["date"].dt.strftime("%Y-%m-%d") + " " + collisions_clean["time"],
    format="%Y-%m-%d %H:%M",
)
collisions_clean["hour"] = collisions_clean["datetime"].dt.hour
collisions_clean["month"] = collisions_clean["date"].dt.month

# --- 3. Country from police force code (see Step 6) --------------------------

force = collisions_clean["police_force"]
collisions_clean["country"] = np.select(
    [force < 60, force.between(60, 63), force == 99],
    ["England", "Wales", "Scotland"],
    default="Unknown",
)

# --- 4. Flags from the police-recorded conditions -----------------------------

# These flags are used later to compare police reports with HadUK-Grid.
# Unknown or missing codes become NaN rather than False, so a missing report
# is never counted as "no rain". Weather code 8 ("Other") is also treated as
# unknown, because it does not say whether it was raining.
weather = collisions_clean["weather_conditions"]
weather_known = ~weather.isin([-1, 8, 9])
collisions_clean["police_reported_rain"] = weather.isin([2, 5]).where(weather_known).astype("boolean")
collisions_clean["police_reported_snow"] = weather.isin([3, 6]).where(weather_known).astype("boolean")

surface = collisions_clean["road_surface_conditions"]
surface_known = ~surface.isin([-1, 9])
collisions_clean["surface_wet"] = surface.isin([2, 5]).where(surface_known).astype("boolean")       # wet/damp or flooded
collisions_clean["surface_ice_snow"] = surface.isin([3, 4]).where(surface_known).astype("boolean")  # snow or frost/ice

# --- 5. Casualty counts by severity and class for each collision ---------------

# crosstab counts casualties per collision for each category, giving one
# row per collision that can be joined back without duplicating collisions.
severity_counts = pd.crosstab(
    casualties["collision_index"], casualties["casualty_severity_label"]
).rename(columns=lambda s: f"casualties_{s.lower()}")

pedestrian_counts = (
    casualties[casualties["casualty_class_label"] == "Pedestrian"]
    .groupby("collision_index").size()
    .rename("casualties_pedestrian")
)

# validate="one_to_one" makes pandas stop with an error if a join would
# duplicate any collision.
collisions_clean = (
    collisions_clean
    .merge(severity_counts, left_on="collision_index", right_index=True, how="left", validate="one_to_one")
    .merge(pedestrian_counts, left_on="collision_index", right_index=True, how="left", validate="one_to_one")
)

# Collisions with no pedestrian casualties get 0 rather than blank.
count_columns = ["casualties_fatal", "casualties_serious", "casualties_slight", "casualties_pedestrian"]
collisions_clean[count_columns] = collisions_clean[count_columns].fillna(0).astype(int)

# --- 6. Keep the columns needed for the analysis --------------------------------

keep_columns = [
    # Identifiers and time
    "collision_index", "date", "datetime", "month", "day_of_week", "hour",
    # Location
    "location_easting_osgr", "location_northing_osgr", "longitude", "latitude",
    "country", "police_force", "local_authority_ons_district", "urban_or_rural_area_label",
    # Road
    "road_type", "speed_limit",
    # Conditions recorded by the police
    "light_conditions_label", "weather_conditions_label", "road_surface_conditions_label",
    "police_reported_rain", "police_reported_snow", "surface_wet", "surface_ice_snow",
    # Outcomes
    "collision_severity", "collision_severity_label", "number_of_vehicles",
    "number_of_casualties", "casualties_fatal", "casualties_serious",
    "casualties_slight", "casualties_pedestrian",
]
collisions_clean = collisions_clean[keep_columns].reset_index(drop=True)

# --- 7. Checks -------------------------------------------------------------------

# Severity counts must add up to the total recorded on each collision.
severity_total = collisions_clean[["casualties_fatal", "casualties_serious", "casualties_slight"]].sum(axis=1)
assert (severity_total == collisions_clean["number_of_casualties"]).all()
assert collisions_clean["collision_index"].is_unique

# Every casualty should be accounted for, apart from those in the
# collisions removed for having no location.
removed_casualties = int(collisions.loc[~has_location, "number_of_casualties"].sum())
assert collisions_clean["number_of_casualties"].sum() == len(casualties) - removed_casualties

print("\nCleaned collisions:", collisions_clean.shape)
print("Casualties in removed collisions:", removed_casualties)
print("Casualties: fatal", collisions_clean["casualties_fatal"].sum(),
      "| serious", collisions_clean["casualties_serious"].sum(),
      "| slight", collisions_clean["casualties_slight"].sum())
print("Police-reported rain (where known):",
      f"{100 * collisions_clean['police_reported_rain'].mean():.2f}% of collisions")
collisions_clean.head()

Collisions before: 101525
Collisions without a location removed: 2
Collisions kept: 101523

Cleaned collisions: (101523, 31)
Casualties in removed collisions: 2
Casualties: fatal 1538 | serious 27757 | slight 98586
Police-reported rain (where known): 11.59% of collisions


,collision_index,date,datetime,month,day_of_week,hour,location_easting_osgr,location_northing_osgr,longitude,latitude,...,surface_wet,surface_ice_snow,collision_severity,collision_severity_label,number_of_vehicles,number_of_casualties,casualties_fatal,casualties_serious,casualties_slight,casualties_pedestrian
0,202517M101925,2025-02-04,2025-02-04 15:00:00,2,3,15,447673.0,519279.0,-1.26420,54.56650,...,False,False,3,Slight,1,1,0,0,1,1
1,202517L205225,2025-05-07,2025-05-07 09:35:00,5,4,9,454611.0,518597.0,-1.15704,54.55967,...,False,False,3,Slight,2,1,0,0,1,0
2,2025070971700,2025-11-27,2025-11-27 18:18:00,11,5,18,354476.0,382656.0,-2.68516,53.33884,...,True,False,3,Slight,2,2,0,0,2,0
3,2025070403274,2025-05-15,2025-05-15 16:30:00,5,5,16,361793.0,393163.0,-2.57655,53.43386,...,False,False,3,Slight,2,1,0,0,1,0
4,2025071000604,2025-12-08,2025-12-08 07:45:00,12,2,7,366083.0,389327.0,-2.51157,53.39967,...,True,False,3,Slight,7,1,0,0,1,0


#### Interpretation of Step 8

The cleaned dataset contains **101,523 collisions × 31 columns**, one row per collision. Only the 2 collisions without a location were removed, together with their 2 casualties.

| Check | Result |
|---|---|
| Collisions kept | 101,523 of 101,525 (99.998%) |
| Casualties kept | 127,881 of 127,883 |
| Casualties by severity | Fatal 1,538; Serious 27,757; Slight 98,586 |
| Severity counts vs `number_of_casualties` | Agree for every collision |
| Duplicate collisions | None |
| Police-reported rain (where weather is known) | 11.59% of collisions |

Each row now has a proper date-time, a location in British National Grid coordinates, the police-recorded conditions as labels and true/false flags, and casualty counts by severity. This is everything needed to attach HadUK-Grid rainfall and temperature in notebook 03.

### Step 9: Save the Cleaned Collision Dataset and Verify It

We save the cleaned dataset as a CSV file, a format readable by any tool and suitable for submission as the final cleaned dataset. CSV does not store column types, so when the file is reloaded we specify them again: identifiers as text, the condition flags as true/false values that can be blank, and dates as date-times.

Reloading and checking the saved file confirms that nothing was lost or changed on the way to disk. Notebook 03 will start from this file.

In [14]:
# Create the output directory if it does not already exist.
output_dir = Path("../data/incident/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save without the pandas index, which is only a row number.
output_path = output_dir / "collisions_clean_2025.csv"
collisions_clean.to_csv(output_path, index=False)

print("Saved dataset:", output_path)
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")

# --- Reload and verify ---

# CSV stores everything as text, so the column types are specified again.
# "boolean" is pandas' true/false type that can also hold blanks (unknown).
flag_columns = ["police_reported_rain", "police_reported_snow", "surface_wet", "surface_ice_snow"]

reloaded = pd.read_csv(
    output_path,
    dtype={
        "collision_index": str,
        "local_authority_ons_district": str,
        **{column: "boolean" for column in flag_columns},
    },
    parse_dates=["date", "datetime"],
)

# Same size, same collisions in the same order, same casualty total.
assert reloaded.shape == collisions_clean.shape
assert reloaded["collision_index"].is_unique
assert reloaded["collision_index"].equals(collisions_clean["collision_index"])
assert reloaded["number_of_casualties"].sum() == collisions_clean["number_of_casualties"].sum()

# Dates survived as real dates within 2025.
assert reloaded["date"].between("2025-01-01", "2025-12-31").all()

# Flags kept both their true/false values and their blanks (unknowns).
for column in flag_columns:
    assert reloaded[column].isna().sum() == collisions_clean[column].isna().sum()
    assert int(reloaded[column].sum()) == int(collisions_clean[column].sum())

# Every collision has a location for the weather join.
assert reloaded["location_easting_osgr"].notna().all()

print("Reloaded shape:", reloaded.shape)
print("Date column type:", reloaded["date"].dtype)
print("All checks on the saved file passed.")

Saved dataset: ..\data\incident\processed\collisions_clean_2025.csv
File size: 20.5 MB
Reloaded shape: (101523, 31)
Date column type: datetime64[ns]
All checks on the saved file passed.
